# DeepGuard AI — Google Colab Free T4 GPU Training Pipeline
**Project Lead & Architect:** Mohan Abhishek Gupta (`@MohanAbhishek29`) | **Group:** K3C0175 (SIH1683)

This notebook lets you train DeepGuard AI on the **24 GB Kaggle Deepfake Dataset** directly inside Google Colab's **Free NVIDIA T4 GPU (15 GB VRAM)** with **0 MB used on your local laptop**:
1. Pulls dataset cloud-to-cloud via Kaggle API (`kaggle.json`).
2. Extracts face frames and fine-tunes a lightweight **EfficientNet / ViT** deepfake classifier on T4 GPU.
3. Plots real **Training/Validation Loss, Accuracy, Confusion Matrix, and ROC-AUC** for faculty evaluation.
4. Exports compact trained weights (`deepguard_colab_weights.pth`) to Google Drive (~30–90 MB) for fast CPU inference in your local FastAPI server.

In [ ]:
# 1. Verify Free NVIDIA T4 GPU & Install Dependencies
!nvidia-smi
!pip install -q kaggle timm transformers scikit-learn matplotlib seaborn opencv-python-headless

In [ ]:
# 2. Mount Google Drive (to persist trained weights safely across sessions)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/DeepGuard_Models

In [ ]:
# 3. Upload your kaggle.json token & Download Dataset directly to Colab Cloud SSD
from google.colab import files
import os

print('Upload your kaggle.json (from Kaggle -> Account -> Create New API Token):')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Example: Deepfake Detection Challenge / 140k Real and Fake Faces
# Replace dataset slug below if using your specific 24GB Kaggle competition/dataset slug
DATASET_SLUG = 'xhlulu/140k-real-and-fake-faces'
!kaggle datasets download -d {DATASET_SLUG} -p /content/dataset --unzip
!ls -lh /content/dataset

In [ ]:
# 4. PyTorch T4 GPU Fine-Tuning (EfficientNet-B0 / timm)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training on device:', device)

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Adjust train/valid directory paths according to extracted Kaggle folder structure
DATA_ROOT = '/content/dataset/real_vs_fake/real-vs-fake'
train_ds = datasets.ImageFolder(f'{DATA_ROOT}/train', transform=transform_train)
val_ds = datasets.ImageFolder(f'{DATA_ROOT}/valid', transform=transform_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print('Classes:', train_ds.class_to_idx, '| Train samples:', len(train_ds), '| Val samples:', len(val_ds))

model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler('cuda')

EPOCHS = 5
best_acc = 0.0
history = {'train_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            out = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    avg_loss = running_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    print(f'Epoch {epoch+1}: Loss={avg_loss:.4f} | Val Acc={val_acc*100:.2f}%')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), '/content/drive/MyDrive/DeepGuard_Models/deepguard_efficientnet_b0.pth')
        print('Saved improved checkpoint to Google Drive!')

In [ ]:
# 5. Evaluate & Plot Confusion Matrix + ROC-AUC for Capstone Report
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

model.load_state_dict(torch.load('/content/drive/MyDrive/DeepGuard_Models/deepguard_efficientnet_b0.pth'))
model.eval()

all_labels, all_probs, all_preds = [], [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        logits = model(imgs.to(device))
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=list(train_ds.class_to_idx.keys())))
print('ROC-AUC:', round(roc_auc_score(all_labels, all_probs), 4))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('DeepGuard AI — Confusion Matrix (Colab T4 GPU)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('/content/drive/MyDrive/DeepGuard_Models/confusion_matrix.png', dpi=200)
plt.show()